In [ ]:
# ================================================================
# 🍅 TOMATO CNN — IMAGE TESTING PREPROCESSING DIAGNOSTIC
# READ-ONLY — NO MODEL/DATASET MODIFICATION
# ================================================================

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

print("=" * 80)
print("🍅 TOMATO CNN — IMAGE TESTING PREPROCESSING DIAGNOSTIC")
print("=" * 80)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version :", tf.__version__)
print("Random seed        :", SEED)


# ================================================================
# 1. GOOGLE DRIVE
# ================================================================

print("\n" + "=" * 80)
print("☁️ GOOGLE DRIVE CHECK")
print("=" * 80)

from google.colab import drive

DRIVE = "/content/drive"
MYDRIVE = "/content/drive/MyDrive"

if not os.path.isdir(MYDRIVE):
    drive.mount(DRIVE)

print("✅ Google Drive accessible.")


# ================================================================
# 2. PATHS
# ================================================================

PROJECT = (
    "/content/drive/MyDrive/"
    "Plant Disease Detection (Computer Vision)"
)

DATASET_PATH = os.path.join(
    PROJECT,
    "3rd Preprocessing",
    "tomato_processed_data.npz"
)

MODEL_PATH = os.path.join(
    PROJECT,
    "6th Trained_Model",
    "tomato_cnn_best.keras"
)

print("\n" + "=" * 80)
print("📁 PATH CHECK")
print("=" * 80)

print("Project :", PROJECT)
print("Dataset :", DATASET_PATH)
print("Model   :", MODEL_PATH)

assert os.path.isdir(PROJECT), "❌ Project folder not found."
assert os.path.isfile(DATASET_PATH), "❌ Tomato dataset not found."
assert os.path.isfile(MODEL_PATH), "❌ Tomato model not found."

print("✅ All required files found.")


# ================================================================
# 3. LOAD DATASET
# ================================================================

print("\n" + "=" * 80)
print("📥 LOADING TOMATO DATASET")
print("=" * 80)

data = np.load(
    DATASET_PATH,
    allow_pickle=True
)

X = data["X"]
y = data["y"]
class_names = data["class_names"]
source = data["source"]

print("X shape         :", X.shape)
print("X dtype         :", X.dtype)
print("y shape         :", y.shape)
print("class_names     :", class_names)
print("source shape    :", source.shape)

assert X.shape == (1800, 224, 224, 3)
assert X.dtype == np.uint8
assert y.shape == (1800,)

print("✅ Dataset loaded correctly.")


# ================================================================
# 4. LOAD MODEL
# ================================================================

print("\n" + "=" * 80)
print("🧠 LOADING FINAL TOMATO CNN")
print("=" * 80)

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)
print("Parameters  :", model.count_params())

assert model.input_shape == (None, 224, 224, 3)
assert model.output_shape == (None, 3)

print("✅ Model loaded successfully.")


# ================================================================
# 5. INSPECT MODEL PREPROCESSING
# ================================================================

print("\n" + "=" * 80)
print("🔍 MODEL PREPROCESSING INSPECTION")
print("=" * 80)

rescaling_layers = []
normalization_layers = []

for i, layer in enumerate(model.layers):

    layer_name = layer.name
    layer_type = layer.__class__.__name__

    print(
        f"{i:02d} | "
        f"{layer_type:<25} | "
        f"{layer_name}"
    )

    if layer_type.lower() == "rescaling":
        rescaling_layers.append(layer)

    if layer_type.lower() == "normalization":
        normalization_layers.append(layer)


print("\n" + "-" * 80)

if rescaling_layers:

    print("⚠️ RESCALING LAYER FOUND")

    for layer in rescaling_layers:

        print("Layer :", layer.name)

        try:
            print("Scale :", layer.scale)
            print("Offset:", layer.offset)
        except:
            pass

    print(
        "\n➡️ The model appears to perform input rescaling internally."
    )

else:

    print("ℹ️ No Rescaling layer found inside the model.")


if normalization_layers:

    print("\n⚠️ NORMALIZATION LAYER FOUND")

    for layer in normalization_layers:
        print("Layer :", layer.name)

else:

    print("\nℹ️ No Normalization layer found.")


# ================================================================
# 6. SELECT EXACT SAME 20-IMAGE TEST SAMPLE
# ================================================================

print("\n" + "=" * 80)
print("🎯 SELECTING 20 TEST IMAGES")
print("=" * 80)

internal_indices = np.where(
    source == "Internal_PlantVillage"
)[0]

external_indices = np.where(
    source == "External_Natural"
)[0]

print("Internal available :", len(internal_indices))
print("External available :", len(external_indices))


rng = np.random.default_rng(SEED)

internal_sample = rng.choice(
    internal_indices,
    size=10,
    replace=False
)

external_sample = rng.choice(
    external_indices,
    size=10,
    replace=False
)

test_indices = np.concatenate(
    [internal_sample, external_sample]
)

print("\nInternal samples :", len(internal_sample))
print("External samples :", len(external_sample))
print("Total samples   :", len(test_indices))

print("\nSelected indices:")
print(test_indices)

X_sample = X[test_indices]
y_sample = y[test_indices]
source_sample = source[test_indices]

print("\nSample shape :", X_sample.shape)
print("Sample dtype :", X_sample.dtype)
print(
    "Raw range    :",
    X_sample.min(),
    "to",
    X_sample.max()
)


# ================================================================
# 7. TEST A — RAW UINT8 [0,255]
# ================================================================

print("\n" + "=" * 80)
print("🧪 TEST A — RAW UINT8 [0,255]")
print("=" * 80)

X_raw = X_sample.astype(np.float32)

print("Input dtype :", X_raw.dtype)
print(
    "Input range :",
    X_raw.min(),
    "to",
    X_raw.max()
)

pred_raw_prob = model.predict(
    X_raw,
    verbose=1
)

pred_raw = np.argmax(
    pred_raw_prob,
    axis=1
)

raw_confidence = np.max(
    pred_raw_prob,
    axis=1
)

raw_correct = (
    pred_raw == y_sample
)

raw_accuracy = np.mean(
    raw_correct
)

print("\nRaw accuracy :", f"{raw_accuracy:.4f}")
print("Raw accuracy :", f"{raw_accuracy * 100:.2f}%")

print("\nRaw prediction distribution:")

for i, class_name in enumerate(class_names):

    count = np.sum(pred_raw == i)

    print(
        f"{class_name:<15}: {count}"
    )


# ================================================================
# 8. TEST B — NORMALIZED FLOAT32 [0,1]
# ================================================================

print("\n" + "=" * 80)
print("🧪 TEST B — NORMALIZED FLOAT32 [0,1]")
print("=" * 80)

X_normalized = (
    X_sample.astype(np.float32) / 255.0
)

print("Input dtype :", X_normalized.dtype)
print(
    "Input range :",
    X_normalized.min(),
    "to",
    X_normalized.max()
)

pred_norm_prob = model.predict(
    X_normalized,
    verbose=1
)

pred_norm = np.argmax(
    pred_norm_prob,
    axis=1
)

norm_confidence = np.max(
    pred_norm_prob,
    axis=1
)

norm_correct = (
    pred_norm == y_sample
)

norm_accuracy = np.mean(
    norm_correct
)

print("\nNormalized accuracy :", f"{norm_accuracy:.4f}")
print(
    "Normalized accuracy :",
    f"{norm_accuracy * 100:.2f}%"
)

print("\nNormalized prediction distribution:")

for i, class_name in enumerate(class_names):

    count = np.sum(pred_norm == i)

    print(
        f"{class_name:<15}: {count}"
    )


# ================================================================
# 9. SIDE-BY-SIDE COMPARISON
# ================================================================

print("\n" + "=" * 80)
print("📊 RAW VS NORMALIZED COMPARISON")
print("=" * 80)

comparison = pd.DataFrame({

    "dataset_index": test_indices,

    "source":
        np.where(
            source_sample == "Internal_PlantVillage",
            "Internal",
            "External"
        ),

    "actual_class":
        class_names[y_sample],

    "raw_prediction":
        class_names[pred_raw],

    "raw_confidence":
        np.round(raw_confidence, 4),

    "raw_correct":
        raw_correct,

    "normalized_prediction":
        class_names[pred_norm],

    "normalized_confidence":
        np.round(norm_confidence, 4),

    "normalized_correct":
        norm_correct
})

print(comparison.to_string(index=False))


# ================================================================
# 10. ACCURACY COMPARISON
# ================================================================

print("\n" + "=" * 80)
print("📈 ACCURACY COMPARISON")
print("=" * 80)

print(
    f"RAW UINT8 [0,255]      : "
    f"{raw_accuracy * 100:.2f}%"
)

print(
    f"NORMALIZED [0,1]      : "
    f"{norm_accuracy * 100:.2f}%"
)


# ================================================================
# 11. AUTOMATIC DIAGNOSTIC
# ================================================================

print("\n" + "=" * 80)
print("🧠 AUTOMATIC DIAGNOSTIC")
print("=" * 80)

difference = abs(
    raw_accuracy - norm_accuracy
)

if rescaling_layers:

    print(
        "⚠️ Model contains an internal Rescaling layer."
    )

    print(
        "➡️ Raw [0,255] input is the expected candidate."
    )

    if raw_accuracy > norm_accuracy:

        print(
            "✅ RAW INPUT PERFORMED BETTER."
        )

    else:

        print(
            "⚠️ Unexpected result: normalized input "
            "performed better."
        )

elif raw_accuracy > norm_accuracy:

    print(
        "⚠️ No Rescaling layer found, but RAW input "
        "performed better."
    )

    print(
        "➡️ This suggests the training pipeline may "
        "have used raw pixel values."
    )

elif norm_accuracy > raw_accuracy:

    print(
        "✅ NORMALIZED [0,1] INPUT PERFORMED BETTER."
    )

    print(
        "➡️ This suggests the model expects "
        "normalized image values."
    )

else:

    print(
        "⚠️ Both preprocessing methods produced "
        "the same accuracy."
    )


# ================================================================
# 12. CHECK WHETHER THE TWO PREDICTIONS ARE IDENTICAL
# ================================================================

same_predictions = np.sum(
    pred_raw == pred_norm
)

print("\n" + "=" * 80)
print("🔬 PREDICTION BEHAVIOR CHECK")
print("=" * 80)

print(
    f"Same predictions : "
    f"{same_predictions} / {len(test_indices)}"
)

print(
    f"Different        : "
    f"{len(test_indices) - same_predictions} / {len(test_indices)}"
)


# ================================================================
# 13. FINAL STATUS
# ================================================================

print("\n" + "=" * 80)
print("🏁 PREPROCESSING DIAGNOSTIC COMPLETE")
print("=" * 80)

print(
    "❗ DO NOT START INTERNET IMAGE TESTING YET."
)

print(
    "First determine which input format matches "
    "the model's training pipeline."
)

print("=" * 80)

🍅 TOMATO CNN — IMAGE TESTING PREPROCESSING DIAGNOSTIC
TensorFlow version : 2.20.0
Random seed        : 42

☁️ GOOGLE DRIVE CHECK
✅ Google Drive accessible.

📁 PATH CHECK
Project : /content/drive/MyDrive/Plant Disease Detection (Computer Vision)
Dataset : /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/tomato_processed_data.npz
Model   : /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/tomato_cnn_best.keras
✅ All required files found.

📥 LOADING TOMATO DATASET
X shape         : (1800, 224, 224, 3)
X dtype         : uint8
y shape         : (1800,)
class_names     : ['Healthy' 'Early_Blight' 'Late_Blight']
source shape    : (1800,)
✅ Dataset loaded correctly.

🧠 LOADING FINAL TOMATO CNN
Input shape : (None, 224, 224, 3)
Output shape: (None, 3)
Parameters  : 2422339
✅ Model loaded successfully.

🔍 MODEL PREPROCESSING INSPECTION
00 | InputLayer                | input_layer_6
01 | Sequential                | tomato_augmenta

In [ ]:
# =============================================================================
# 6. LOAD EXACT FINAL EVALUATION TEST SET
# =============================================================================

print("\n" + "=" * 80)
print("🎯 RECOVERING EXACT FINAL TEST SET")
print("=" * 80)

previous_results = pd.read_csv(PREDICTION_CSV)

print("Previous prediction rows :", len(previous_results))

print("\nCSV columns:")
print(list(previous_results.columns))


# ------------------------------------------------
# YOUR ACTUAL CSV USES "original_index"
# ------------------------------------------------

if "original_index" not in previous_results.columns:

    raise ValueError(
        "\n❌ 'original_index' column not found in "
        "tomato_test_predictions.csv."
    )

test_indices = (
    previous_results["original_index"]
    .astype(int)
    .to_numpy()
)

print("\nIndex column used : original_index")
print("Test images recovered :", len(test_indices))


# ------------------------------------------------
# EXACTLY 270 TEST IMAGES EXPECTED
# ------------------------------------------------

if len(test_indices) != 270:

    raise ValueError(
        f"\n❌ Expected exactly 270 test images, "
        f"but found {len(test_indices)}."
    )


# ------------------------------------------------
# CHECK DUPLICATES
# ------------------------------------------------

if len(np.unique(test_indices)) != 270:

    raise ValueError(
        "\n❌ Duplicate dataset indices detected "
        "inside the saved test prediction CSV."
    )


# ------------------------------------------------
# CHECK RANGE
# ------------------------------------------------

if (
    np.min(test_indices) < 0
    or np.max(test_indices) >= len(X)
):

    raise ValueError(
        "\n❌ One or more test indices are outside "
        "the tomato dataset range."
    )


print("Minimum dataset index :", np.min(test_indices))
print("Maximum dataset index :", np.max(test_indices))

print("✅ EXACT 270-image test set recovered.")
print("✅ Indices are unique.")
print("✅ Indices are valid.")


🎯 RECOVERING EXACT FINAL TEST SET
Previous prediction rows : 270

CSV columns:
['original_index', 'actual_class', 'predicted_class', 'confidence', 'correct', 'source', 'probability_healthy', 'probability_early_blight', 'probability_late_blight']

Index column used : original_index
Test images recovered : 270
Minimum dataset index : 2
Maximum dataset index : 1792
✅ EXACT 270-image test set recovered.
✅ Indices are unique.
✅ Indices are valid.


In [ ]:
# =============================================================================
# 🍅 TOMATO CNN — COMPLETE PREPROCESSING FORENSIC EXECUTION
# =============================================================================
# PURPOSE:
#   1. Load exact final 270-image test set
#   2. Load final Tomato CNN
#   3. Test RAW [0,255]
#   4. Test NORMALIZED [0,1]
#   5. Test MobileNetV2 [-1,1]
#   6. Compare against original 97.78% evaluation
#
# ⚠️ READ-ONLY
# ⚠️ NO MODEL MODIFICATION
# ⚠️ NO DATASET MODIFICATION
# ⚠️ NO FILE DELETION
# =============================================================================

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 80)
print("🍅 TOMATO CNN — PREPROCESSING FORENSIC EXECUTION")
print("=" * 80)

print("TensorFlow version :", tf.__version__)
print("Random seed        :", SEED)


# =============================================================================
# 1. GOOGLE DRIVE
# =============================================================================

print("\n" + "=" * 80)
print("☁️ GOOGLE DRIVE CHECK")
print("=" * 80)

MYDRIVE = "/content/drive/MyDrive"

if not os.path.isdir(MYDRIVE):

    from google.colab import drive
    drive.mount("/content/drive")

print("✅ Google Drive accessible.")


# =============================================================================
# 2. PATHS
# =============================================================================

PROJECT = (
    "/content/drive/MyDrive/"
    "Plant Disease Detection (Computer Vision)"
)

DATASET_PATH = os.path.join(
    PROJECT,
    "3rd Preprocessing",
    "tomato_processed_data.npz"
)

MODEL_PATH = os.path.join(
    PROJECT,
    "6th Trained_Model",
    "tomato_cnn_best.keras"
)

PREDICTION_CSV = os.path.join(
    PROJECT,
    "5th Model_Evaluation",
    "tomato_test_predictions.csv"
)

DIAGNOSTIC_DIR = os.path.join(
    PROJECT,
    "5th Model_Evaluation",
    "Tomato_Image_Testing"
)

os.makedirs(DIAGNOSTIC_DIR, exist_ok=True)


# =============================================================================
# 3. FILE CHECK
# =============================================================================

print("\n" + "=" * 80)
print("📁 FILE CHECK")
print("=" * 80)

required_files = {
    "Dataset": DATASET_PATH,
    "Model": MODEL_PATH,
    "Previous evaluation CSV": PREDICTION_CSV
}

for name, path in required_files.items():

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"\n❌ {name} not found:\n{path}"
        )

    print(f"✅ {name} found")


# =============================================================================
# 4. LOAD DATASET
# =============================================================================

print("\n" + "=" * 80)
print("📥 LOADING TOMATO DATASET")
print("=" * 80)

data = np.load(
    DATASET_PATH,
    allow_pickle=True
)

X = data["X"]
y = data["y"]
class_names = data["class_names"]

print("X shape         :", X.shape)
print("X dtype         :", X.dtype)
print("y shape         :", y.shape)
print("class_names     :", class_names)

if X.shape != (1800, 224, 224, 3):
    raise ValueError(f"Unexpected X shape: {X.shape}")

if y.shape != (1800,):
    raise ValueError(f"Unexpected y shape: {y.shape}")

print("✅ Dataset loaded and verified.")


# =============================================================================
# 5. RECOVER EXACT 270 TEST IMAGES
# =============================================================================

print("\n" + "=" * 80)
print("🎯 RECOVERING EXACT FINAL TEST SET")
print("=" * 80)

previous_results = pd.read_csv(PREDICTION_CSV)

print("Previous prediction rows :", len(previous_results))

print("CSV columns:")
print(list(previous_results.columns))

if "original_index" not in previous_results.columns:

    raise ValueError(
        "\n❌ 'original_index' column not found."
    )

test_indices = (
    previous_results["original_index"]
    .astype(int)
    .to_numpy()
)

if len(test_indices) != 270:

    raise ValueError(
        f"\n❌ Expected 270 test images, "
        f"found {len(test_indices)}."
    )

if len(np.unique(test_indices)) != 270:

    raise ValueError(
        "\n❌ Duplicate test indices detected."
    )

if (
    np.min(test_indices) < 0
    or np.max(test_indices) >= len(X)
):

    raise ValueError(
        "\n❌ Invalid dataset index detected."
    )

print("Index column          : original_index")
print("Test images recovered :", len(test_indices))
print("Minimum index         :", np.min(test_indices))
print("Maximum index         :", np.max(test_indices))

print("✅ EXACT 270-image test set recovered.")


# =============================================================================
# 6. PREPARE EXACT TEST SET
# =============================================================================

X_test_raw = X[test_indices]
y_test = y[test_indices]

print("\nTest shape :", X_test_raw.shape)
print("Test dtype :", X_test_raw.dtype)
print(
    "Raw range  :",
    X_test_raw.min(),
    "to",
    X_test_raw.max()
)

if X_test_raw.shape != (270, 224, 224, 3):

    raise ValueError(
        f"Unexpected test shape: {X_test_raw.shape}"
    )

print("✅ Exact test data ready.")


# =============================================================================
# 7. LOAD FINAL TOMATO MODEL
# =============================================================================

print("\n" + "=" * 80)
print("🧠 LOADING FINAL TOMATO CNN")
print("=" * 80)

# IMPORTANT:
# The model must be loaded before model.predict().
# This prevents the previous NameError.

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)
print("Parameters  :", model.count_params())

if model.input_shape != (None, 224, 224, 3):

    raise ValueError(
        f"Unexpected model input: {model.input_shape}"
    )

if model.output_shape != (None, 3):

    raise ValueError(
        f"Unexpected model output: {model.output_shape}"
    )

print("✅ Final Tomato CNN loaded successfully.")


# =============================================================================
# 8. PREPARE THREE INPUT FORMATS
# =============================================================================

print("\n" + "=" * 80)
print("🧪 PREPARING THREE INPUT FORMATS")
print("=" * 80)


# A — RAW [0,255]
X_raw = X_test_raw.astype(np.float32)


# B — NORMALIZED [0,1]
X_01 = (
    X_test_raw.astype(np.float32)
    / 255.0
)


# C — MobileNetV2 [-1,1]
X_m11 = (
    X_test_raw.astype(np.float32)
    / 127.5
) - 1.0


print("\nRAW [0,255]")
print("dtype :", X_raw.dtype)
print(
    "range :",
    X_raw.min(),
    "to",
    X_raw.max()
)

print("\nNORMALIZED [0,1]")
print("dtype :", X_01.dtype)
print(
    "range :",
    X_01.min(),
    "to",
    X_01.max()
)

print("\nMobileNetV2 [-1,1]")
print("dtype :", X_m11.dtype)
print(
    "range :",
    X_m11.min(),
    "to",
    X_m11.max()
)

print("\n✅ Three input formats prepared.")


# =============================================================================
# 9. TEST A — RAW
# =============================================================================

print("\n" + "=" * 80)
print("🚀 TEST A — RAW [0,255]")
print("=" * 80)

prob_raw = model.predict(
    X_raw,
    batch_size=32,
    verbose=1
)

pred_raw = np.argmax(
    prob_raw,
    axis=1
)

raw_accuracy = np.mean(
    pred_raw == y_test
)

print(
    f"\nRAW accuracy : "
    f"{raw_accuracy:.4f} "
    f"({raw_accuracy * 100:.2f}%)"
)


# =============================================================================
# 10. TEST B — NORMALIZED
# =============================================================================

print("\n" + "=" * 80)
print("🚀 TEST B — NORMALIZED [0,1]")
print("=" * 80)

prob_01 = model.predict(
    X_01,
    batch_size=32,
    verbose=1
)

pred_01 = np.argmax(
    prob_01,
    axis=1
)

accuracy_01 = np.mean(
    pred_01 == y_test
)

print(
    f"\n[0,1] accuracy : "
    f"{accuracy_01:.4f} "
    f"({accuracy_01 * 100:.2f}%)"
)


# =============================================================================
# 11. TEST C — MOBILENETV2
# =============================================================================

print("\n" + "=" * 80)
print("🚀 TEST C — MOBILENETV2 [-1,1]")
print("=" * 80)

prob_m11 = model.predict(
    X_m11,
    batch_size=32,
    verbose=1
)

pred_m11 = np.argmax(
    prob_m11,
    axis=1
)

accuracy_m11 = np.mean(
    pred_m11 == y_test
)

print(
    f"\n[-1,1] accuracy : "
    f"{accuracy_m11:.4f} "
    f"({accuracy_m11 * 100:.2f}%)"
)


# =============================================================================
# 12. COMPARE PREDICTIONS WITH ORIGINAL SAVED EVALUATION
# =============================================================================

print("\n" + "=" * 80)
print("📊 COMPARING WITH ORIGINAL FINAL EVALUATION")
print("=" * 80)

# Original predictions from saved CSV
original_pred = (
    previous_results["predicted_class"]
    .astype(str)
    .to_numpy()
)

original_actual = (
    previous_results["actual_class"]
    .astype(str)
    .to_numpy()
)

# Convert current numeric predictions to class names
raw_names = np.array([
    str(class_names[i])
    for i in pred_raw
])

names_01 = np.array([
    str(class_names[i])
    for i in pred_01
])

names_m11 = np.array([
    str(class_names[i])
    for i in pred_m11
])


raw_match = np.sum(
    raw_names == original_pred
)

accuracy_01_match = np.sum(
    names_01 == original_pred
)

m11_match = np.sum(
    names_m11 == original_pred
)


print("\nOriginal saved evaluation:")
print("Accuracy : 97.78%")
print("Correct  : 264 / 270")

print("\nPrediction agreement with original CSV:")

print(
    f"RAW [0,255] : "
    f"{raw_match}/270 "
    f"({raw_match / 270 * 100:.2f}%)"
)

print(
    f"[0,1]        : "
    f"{accuracy_01_match}/270 "
    f"({accuracy_01_match / 270 * 100:.2f}%)"
)

print(
    f"[-1,1]       : "
    f"{m11_match}/270 "
    f"({m11_match / 270 * 100:.2f}%)"
)


# =============================================================================
# 13. ACCURACY SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("📈 PREPROCESSING ACCURACY SUMMARY")
print("=" * 80)

summary = pd.DataFrame({

    "input_format": [
        "RAW [0,255]",
        "NORMALIZED [0,1]",
        "MobileNetV2 [-1,1]"
    ],

    "accuracy": [
        raw_accuracy,
        accuracy_01,
        accuracy_m11
    ],

    "accuracy_percent": [
        raw_accuracy * 100,
        accuracy_01 * 100,
        accuracy_m11 * 100
    ],

    "agreement_with_original": [
        raw_match,
        accuracy_01_match,
        m11_match
    ]

})

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# =============================================================================
# 14. FIND BEST MATCH TO ORIGINAL EVALUATION
# =============================================================================

print("\n" + "=" * 80)
print("🏆 PREPROCESSING FORENSIC DECISION")
print("=" * 80)

agreement_scores = {
    "RAW_0_255": raw_match,
    "NORMALIZED_0_1": accuracy_01_match,
    "MOBILENETV2_MINUS1_PLUS1": m11_match
}

best_format = max(
    agreement_scores,
    key=agreement_scores.get
)

best_match = agreement_scores[best_format]


print("Best match format :", best_format)
print(
    "Prediction agreement :",
    f"{best_match}/270",
    f"({best_match / 270 * 100:.2f}%)"
)


# =============================================================================
# 15. FINAL DECISION
# =============================================================================

print("\n" + "=" * 80)
print("🔐 FINAL INPUT CONTRACT CHECK")
print("=" * 80)

if best_match == 270:

    print("🎯 PERFECT MATCH FOUND.")
    print(
        f"Official preprocessing format: {best_format}"
    )
    print(
        "All 270 predictions exactly match "
        "the original final evaluation."
    )

elif best_match >= 265:

    print("✅ VERY STRONG MATCH FOUND.")
    print(
        f"Best preprocessing format: {best_format}"
    )
    print(
        "Further verification may be required "
        "before locking the application pipeline."
    )

else:

    print("⚠️ NO EXACT MATCH FOUND.")

    print(
        "Do NOT lock the application preprocessing yet."
    )

    print(
        "The original evaluation pipeline requires "
        "additional forensic inspection."
    )


# =============================================================================
# 16. SAVE DIAGNOSTIC RESULT
# =============================================================================

diagnostic = {

    "tensorflow_version": tf.__version__,

    "model": MODEL_PATH,

    "dataset": DATASET_PATH,

    "exact_test_images": 270,

    "original_evaluation_accuracy": 0.9778,

    "original_correct": 264,

    "original_incorrect": 6,

    "raw_0_255_accuracy":
        float(raw_accuracy),

    "normalized_0_1_accuracy":
        float(accuracy_01),

    "mobilenetv2_minus1_plus1_accuracy":
        float(accuracy_m11),

    "raw_agreement_with_original":
        int(raw_match),

    "normalized_agreement_with_original":
        int(accuracy_01_match),

    "mobilenetv2_agreement_with_original":
        int(m11_match),

    "best_matching_format":
        best_format,

    "best_prediction_agreement":
        int(best_match),

    "model_modified": False,

    "dataset_modified": False

}

diagnostic_path = os.path.join(
    DIAGNOSTIC_DIR,
    "tomato_preprocessing_forensic_result.json"
)

with open(
    diagnostic_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        diagnostic,
        f,
        indent=4
    )


# =============================================================================
# 17. FINAL STATUS
# =============================================================================

print("\n" + "=" * 80)
print("🏁 TOMATO PREPROCESSING FORENSIC CHECK COMPLETE")
print("=" * 80)

print("Diagnostic saved:")
print(diagnostic_path)

print("\n✅ Model unchanged.")
print("✅ Dataset unchanged.")
print("✅ No files deleted.")

if best_match == 270:

    print(
        "\n🎯 TOMATO PREPROCESSING CONTRACT "
        "CAN NOW BE LOCKED."
    )

else:

    print(
        "\n⚠️ PREPROCESSING CONTRACT "
        "IS NOT YET LOCKED."
    )

print("=" * 80)

🍅 TOMATO CNN — PREPROCESSING FORENSIC EXECUTION
TensorFlow version : 2.20.0
Random seed        : 42

☁️ GOOGLE DRIVE CHECK
✅ Google Drive accessible.

📁 FILE CHECK
✅ Dataset found
✅ Model found
✅ Previous evaluation CSV found

📥 LOADING TOMATO DATASET
X shape         : (1800, 224, 224, 3)
X dtype         : uint8
y shape         : (1800,)
class_names     : ['Healthy' 'Early_Blight' 'Late_Blight']
✅ Dataset loaded and verified.

🎯 RECOVERING EXACT FINAL TEST SET
Previous prediction rows : 270
CSV columns:
['original_index', 'actual_class', 'predicted_class', 'confidence', 'correct', 'source', 'probability_healthy', 'probability_early_blight', 'probability_late_blight']
Index column          : original_index
Test images recovered : 270
Minimum index         : 2
Maximum index         : 1792
✅ EXACT 270-image test set recovered.

Test shape : (270, 224, 224, 3)
Test dtype : uint8
Raw range  : 0 to 255
✅ Exact test data ready.

🧠 LOADING FINAL TOMATO CNN
Input shape : (None, 224, 224, 3)
Out

In [ ]:
# =============================================================================
# 🍅 TOMATO CNN — FINAL REAL-WORLD / INTERNET IMAGE TEST
# =============================================================================
# IMPORTANT:
# - Independent images only
# - NOT used during training
# - NOT added to the training dataset
# - Model is READ-ONLY
# - Dataset is READ-ONLY
# - Official Tomato preprocessing:
#       RGB → 224×224 → float32 → 0–255
# =============================================================================

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from PIL import Image

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 80)
print("🍅 TOMATO CNN — FINAL REAL-WORLD IMAGE TEST")
print("=" * 80)

print("TensorFlow version :", tf.__version__)
print("Random seed        :", SEED)


# =============================================================================
# 1. GOOGLE DRIVE
# =============================================================================

print("\n" + "=" * 80)
print("☁️ GOOGLE DRIVE CHECK")
print("=" * 80)

MYDRIVE = "/content/drive/MyDrive"

if not os.path.isdir(MYDRIVE):
    from google.colab import drive
    drive.mount("/content/drive")

print("✅ Google Drive accessible.")


# =============================================================================
# 2. PATHS
# =============================================================================

PROJECT = (
    "/content/drive/MyDrive/"
    "Plant Disease Detection (Computer Vision)"
)

MODEL_PATH = os.path.join(
    PROJECT,
    "6th Trained_Model",
    "tomato_cnn_best.keras"
)

REAL_WORLD_DIR = os.path.join(
    PROJECT,
    "7th Test_Images",
    "Tomato_RealWorld"
)

OUTPUT_DIR = os.path.join(
    PROJECT,
    "5th Model_Evaluation",
    "Tomato_Image_Testing"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


print("\nProject:")
print(PROJECT)

print("\nModel:")
print(MODEL_PATH)

print("\nReal-world test directory:")
print(REAL_WORLD_DIR)


# =============================================================================
# 3. MODEL CHECK
# =============================================================================

print("\n" + "=" * 80)
print("🧠 LOADING FINAL TOMATO CNN")
print("=" * 80)

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(
        f"\n❌ Tomato model not found:\n{MODEL_PATH}"
    )

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)
print("Parameters  :", model.count_params())

assert model.input_shape == (None, 224, 224, 3)
assert model.output_shape == (None, 3)

print("✅ Final Tomato CNN loaded.")


# =============================================================================
# 4. CLASS DEFINITIONS
# =============================================================================

CLASS_NAMES = [
    "Healthy",
    "Early_Blight",
    "Late_Blight"
]

CLASS_TO_INDEX = {
    name: i
    for i, name in enumerate(CLASS_NAMES)
}


# =============================================================================
# 5. CHECK REAL-WORLD FOLDER
# =============================================================================

print("\n" + "=" * 80)
print("📁 CHECKING REAL-WORLD TEST IMAGES")
print("=" * 80)

if not os.path.isdir(REAL_WORLD_DIR):

    raise FileNotFoundError(
        f"""
❌ Real-world image folder not found:

{REAL_WORLD_DIR}

Create this structure first:

7th Test_Images/
└── Tomato_RealWorld/
    ├── Healthy/
    ├── Early_Blight/
    └── Late_Blight/
"""
    )

print("✅ Real-world test directory found.")


# =============================================================================
# 6. COLLECT IMAGE FILES
# =============================================================================

VALID_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
)

records = []

print("\n" + "=" * 80)
print("🔎 SCANNING INTERNET / REAL-WORLD IMAGES")
print("=" * 80)

for class_name in CLASS_NAMES:

    class_dir = os.path.join(
        REAL_WORLD_DIR,
        class_name
    )

    if not os.path.isdir(class_dir):

        print(
            f"⚠️ Missing class folder: {class_name}"
        )

        continue

    files = sorted(
        [
            f
            for f in os.listdir(class_dir)
            if f.lower().endswith(VALID_EXTENSIONS)
        ]
    )

    print(
        f"{class_name:<15} : {len(files)} images"
    )

    for filename in files:

        records.append({

            "filepath": os.path.join(
                class_dir,
                filename
            ),

            "actual_class": class_name
        })


# =============================================================================
# 7. BASIC TEST SET VALIDATION
# =============================================================================

print("\n" + "=" * 80)
print("📊 TEST SET VALIDATION")
print("=" * 80)

if len(records) == 0:

    raise ValueError(
        "\n❌ No images found."
    )

records_df = pd.DataFrame(records)

print(
    "Total real-world images :",
    len(records_df)
)

print("\nClass distribution:")

print(
    records_df["actual_class"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)


for class_name in CLASS_NAMES:

    count = int(
        np.sum(
            records_df["actual_class"]
            == class_name
        )
    )

    if count == 0:

        raise ValueError(
            f"\n❌ No images found for {class_name}."
        )


print(
    "\n✅ All three Tomato classes represented."
)


# =============================================================================
# 8. LOAD AND PREPROCESS IMAGES
# =============================================================================

print("\n" + "=" * 80)
print("🧪 PREPARING REAL-WORLD IMAGES")
print("=" * 80)

images = []
valid_records = []

for _, row in records_df.iterrows():

    filepath = row["filepath"]

    try:

        img = Image.open(filepath)

        # Force RGB
        img = img.convert("RGB")

        # Official model input size
        img = img.resize(
            (224, 224),
            Image.Resampling.BILINEAR
        )

        # IMPORTANT:
        # Keep pixel values 0–255.
        img_array = np.asarray(
            img,
            dtype=np.float32
        )

        images.append(img_array)

        valid_records.append(row)

    except Exception as e:

        print(
            f"⚠️ Could not read {filepath}"
        )

        print(
            "   Reason:",
            e
        )


if len(images) == 0:

    raise ValueError(
        "\n❌ No readable images."
    )


X_real = np.stack(images)

results_base = pd.DataFrame(
    valid_records
)

print("\nPrepared images :", len(X_real))
print("Shape           :", X_real.shape)
print("Dtype           :", X_real.dtype)
print(
    "Pixel range     :",
    X_real.min(),
    "to",
    X_real.max()
)

assert X_real.shape[1:] == (
    224,
    224,
    3
)

assert X_real.dtype == np.float32

print(
    "✅ Official Tomato preprocessing applied."
)


# =============================================================================
# 9. RUN MODEL
# =============================================================================

print("\n" + "=" * 80)
print("🚀 RUNNING REAL-WORLD PREDICTIONS")
print("=" * 80)

probabilities = model.predict(
    X_real,
    batch_size=16,
    verbose=1
)

predicted_indices = np.argmax(
    probabilities,
    axis=1
)

predicted_classes = [
    CLASS_NAMES[i]
    for i in predicted_indices
]

confidence = np.max(
    probabilities,
    axis=1
)


# =============================================================================
# 10. BUILD RESULTS
# =============================================================================

results = results_base.copy()

results["predicted_class"] = predicted_classes

results["confidence"] = confidence

results["correct"] = (
    results["actual_class"]
    == results["predicted_class"]
)

results["probability_healthy"] = (
    probabilities[:, 0]
)

results["probability_early_blight"] = (
    probabilities[:, 1]
)

results["probability_late_blight"] = (
    probabilities[:, 2]
)


# =============================================================================
# 11. DISPLAY IMAGE-BY-IMAGE RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("📋 IMAGE-BY-IMAGE RESULTS")
print("=" * 80)

display_columns = [
    "actual_class",
    "predicted_class",
    "confidence",
    "correct"
]

print(
    results[display_columns]
    .to_string(index=False)
)


# =============================================================================
# 12. OVERALL ACCURACY
# =============================================================================

correct_count = int(
    results["correct"].sum()
)

total_count = len(results)

incorrect_count = (
    total_count
    - correct_count
)

accuracy = (
    correct_count
    / total_count
)


print("\n" + "=" * 80)
print("📈 REAL-WORLD TEST RESULTS")
print("=" * 80)

print(
    f"Total images      : {total_count}"
)

print(
    f"Correct           : {correct_count}"
)

print(
    f"Incorrect         : {incorrect_count}"
)

print(
    f"Accuracy          : "
    f"{accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)


# =============================================================================
# 13. CLASS-WISE ACCURACY
# =============================================================================

print("\n" + "=" * 80)
print("📊 CLASS-WISE REAL-WORLD PERFORMANCE")
print("=" * 80)

for class_name in CLASS_NAMES:

    class_results = results[
        results["actual_class"]
        == class_name
    ]

    class_correct = int(
        class_results["correct"].sum()
    )

    class_total = len(
        class_results
    )

    class_accuracy = (
        class_correct / class_total
    )

    print(
        f"{class_name:<15} : "
        f"{class_correct}/{class_total} "
        f"("
        f"{class_accuracy * 100:.2f}%"
        f")"
    )


# =============================================================================
# 14. CONFIDENCE
# =============================================================================

print("\n" + "=" * 80)
print("🎯 CONFIDENCE ANALYSIS")
print("=" * 80)

print(
    "Average confidence :",
    f"{results['confidence'].mean():.4f}"
)

print(
    "Minimum confidence :",
    f"{results['confidence'].min():.4f}"
)

print(
    "Maximum confidence :",
    f"{results['confidence'].max():.4f}"
)


# =============================================================================
# 15. MISCLASSIFICATIONS
# =============================================================================

print("\n" + "=" * 80)
print("🔍 MISCLASSIFICATION CHECK")
print("=" * 80)

wrong = results[
    results["correct"] == False
]

print(
    "Misclassified images :",
    len(wrong)
)

if len(wrong) > 0:

    print("\nMisclassified cases:")

    print(
        wrong[
            [
                "actual_class",
                "predicted_class",
                "confidence"
            ]
        ].to_string(index=False)
    )

else:

    print(
        "🎉 All real-world images "
        "classified correctly."
    )


# =============================================================================
# 16. SAVE RESULTS
# =============================================================================

RESULT_CSV = os.path.join(
    OUTPUT_DIR,
    "tomato_real_world_image_test.csv"
)

SUMMARY_JSON = os.path.join(
    OUTPUT_DIR,
    "tomato_real_world_image_test_summary.json"
)

results.to_csv(
    RESULT_CSV,
    index=False
)

summary = {

    "model": MODEL_PATH,

    "test_type":
        "Independent real-world / internet images",

    "total_images":
        int(total_count),

    "correct":
        int(correct_count),

    "incorrect":
        int(incorrect_count),

    "accuracy":
        float(accuracy),

    "accuracy_percent":
        float(accuracy * 100),

    "average_confidence":
        float(
            results["confidence"].mean()
        ),

    "preprocessing":
        {
            "color": "RGB",
            "resize": "224x224",
            "dtype": "float32",
            "pixel_range": "0-255",
            "normalization": False
        },

    "model_modified": False,

    "training_dataset_modified": False
}


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


print("\n" + "=" * 80)
print("💾 RESULTS SAVED")
print("=" * 80)

print("CSV:")
print(RESULT_CSV)

print("\nSummary:")
print(SUMMARY_JSON)


# =============================================================================
# 17. FINAL STATUS
# =============================================================================

print("\n" + "=" * 80)
print("🏁 TOMATO REAL-WORLD IMAGE TEST COMPLETE")
print("=" * 80)

print(
    f"Images tested : {total_count}"
)

print(
    f"Accuracy      : "
    f"{accuracy * 100:.2f}%"
)

print(
    f"Correct       : {correct_count}"
)

print(
    f"Incorrect     : {incorrect_count}"
)

print("\n✅ Model unchanged.")
print("✅ Training dataset unchanged.")
print("✅ No training performed.")
print("✅ No files deleted.")

print("\n" + "=" * 80)

🍅 TOMATO CNN — FINAL REAL-WORLD IMAGE TEST
TensorFlow version : 2.20.0
Random seed        : 42

☁️ GOOGLE DRIVE CHECK
✅ Google Drive accessible.

Project:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)

Model:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/tomato_cnn_best.keras

Real-world test directory:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/7th Test_Images/Tomato_RealWorld

🧠 LOADING FINAL TOMATO CNN
Input shape : (None, 224, 224, 3)
Output shape: (None, 3)
Parameters  : 2422339
✅ Final Tomato CNN loaded.

📁 CHECKING REAL-WORLD TEST IMAGES
✅ Real-world test directory found.

🔎 SCANNING INTERNET / REAL-WORLD IMAGES
Healthy         : 5 images
Early_Blight    : 5 images
Late_Blight     : 6 images

📊 TEST SET VALIDATION
Total real-world images : 16

Class distribution:
actual_class
Healthy         5
Early_Blight    5
Late_Blight     6
Name: count, dtype: int64

✅ All three Tomato classes represented.

🧪 PREP

In [ ]:
# =============================================================================
# 🍅 TOMATO CNN — INTERNAL RAW DATA SUBSET TEST
# =============================================================================
# PURPOSE:
#   1. Load original tomato raw/processed dataset
#   2. Inspect internal data
#   3. Exclude the exact 270-image final evaluation set
#   4. Randomly select 15 INTERNAL images
#      - 5 Healthy
#      - 5 Early_Blight
#      - 5 Late_Blight
#   5. Test using OFFICIAL preprocessing:
#        RGB → 224×224 → float32 → 0–255
#   6. Compare internal performance
#
# ⚠️ READ-ONLY
# ⚠️ NO MODEL MODIFICATION
# ⚠️ NO DATASET MODIFICATION
# ⚠️ NO TRAINING
# ⚠️ NO FILE DELETION
# =============================================================================

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 80)
print("🍅 TOMATO CNN — INTERNAL RAW DATA SUBSET TEST")
print("=" * 80)

print("TensorFlow version :", tf.__version__)
print("Random seed        :", SEED)


# =============================================================================
# 1. GOOGLE DRIVE
# =============================================================================

print("\n" + "=" * 80)
print("☁️ GOOGLE DRIVE CHECK")
print("=" * 80)

MYDRIVE = "/content/drive/MyDrive"

if not os.path.isdir(MYDRIVE):
    from google.colab import drive
    drive.mount("/content/drive")

print("✅ Google Drive accessible.")


# =============================================================================
# 2. PATHS
# =============================================================================

PROJECT = (
    "/content/drive/MyDrive/"
    "Plant Disease Detection (Computer Vision)"
)

DATASET_PATH = os.path.join(
    PROJECT,
    "3rd Preprocessing",
    "tomato_processed_data.npz"
)

MODEL_PATH = os.path.join(
    PROJECT,
    "6th Trained_Model",
    "tomato_cnn_best.keras"
)

PREDICTION_CSV = os.path.join(
    PROJECT,
    "5th Model_Evaluation",
    "tomato_test_predictions.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT,
    "5th Model_Evaluation",
    "Tomato_Image_Testing"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# =============================================================================
# 3. FILE CHECK
# =============================================================================

print("\n" + "=" * 80)
print("📁 FILE CHECK")
print("=" * 80)

required_files = {
    "Tomato dataset": DATASET_PATH,
    "Tomato CNN": MODEL_PATH,
    "Final evaluation CSV": PREDICTION_CSV
}

for name, path in required_files.items():

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"\n❌ {name} not found:\n{path}"
        )

    print(f"✅ {name} found")


# =============================================================================
# 4. LOAD INTERNAL DATASET
# =============================================================================

print("\n" + "=" * 80)
print("📥 LOADING INTERNAL TOMATO DATASET")
print("=" * 80)

data = np.load(
    DATASET_PATH,
    allow_pickle=True
)

X = data["X"]
y = data["y"]
class_names = data["class_names"]
source = data["source"]

print("X shape         :", X.shape)
print("X dtype         :", X.dtype)
print("y shape         :", y.shape)
print("source shape    :", source.shape)
print("class_names     :", class_names)

print(
    "Raw pixel range :",
    X.min(),
    "to",
    X.max()
)

print("\nSource distribution:")

print(
    pd.Series(source)
    .value_counts()
)

print("\nClass distribution:")

for i, class_name in enumerate(class_names):

    count = int(
        np.sum(y == i)
    )

    print(
        f"{class_name:<15} : {count}"
    )

print("\n✅ Internal dataset loaded.")


# =============================================================================
# 5. RECOVER FINAL 270 EVALUATION INDICES
# =============================================================================
# IMPORTANT:
# We exclude these indices so this test does NOT reuse the final evaluation set.

print("\n" + "=" * 80)
print("🔒 EXCLUDING FINAL 270 EVALUATION IMAGES")
print("=" * 80)

previous_results = pd.read_csv(
    PREDICTION_CSV
)

if "original_index" not in previous_results.columns:

    raise ValueError(
        "\n❌ original_index column not found."
    )

final_test_indices = set(
    previous_results[
        "original_index"
    ].astype(int)
)

print(
    "Final evaluation images :",
    len(final_test_indices)
)

print(
    "These images will NOT be sampled."
)

print("✅ Final evaluation set excluded.")


# =============================================================================
# 6. IDENTIFY AVAILABLE INTERNAL DATA
# =============================================================================

print("\n" + "=" * 80)
print("🔎 IDENTIFYING AVAILABLE INTERNAL IMAGES")
print("=" * 80)

available_indices = []

for idx in range(len(X)):

    if idx not in final_test_indices:

        available_indices.append(idx)

available_indices = np.array(
    available_indices,
    dtype=int
)

print(
    "Total dataset images      :",
    len(X)
)

print(
    "Final evaluation images   :",
    len(final_test_indices)
)

print(
    "Remaining internal images :",
    len(available_indices)
)


# =============================================================================
# 7. SELECT 5 IMAGES PER CLASS
# =============================================================================

print("\n" + "=" * 80)
print("🎯 SELECTING INTERNAL TEST SUBSET")
print("=" * 80)

rng = np.random.default_rng(
    SEED
)

SELECT_PER_CLASS = 5

selected_indices = []

for class_index, class_name in enumerate(class_names):

    class_available = available_indices[
        y[available_indices] == class_index
    ]

    print(
        f"{class_name:<15} available : "
        f"{len(class_available)}"
    )

    if len(class_available) < SELECT_PER_CLASS:

        raise ValueError(
            f"Not enough internal images for {class_name}."
        )

    selected = rng.choice(
        class_available,
        size=SELECT_PER_CLASS,
        replace=False
    )

    selected_indices.extend(
        selected.tolist()
    )


selected_indices = np.array(
    selected_indices,
    dtype=int
)

# Shuffle selected images
rng.shuffle(
    selected_indices
)

print(
    "\nSelected internal images :",
    len(selected_indices)
)

print(
    "Selected indices:"
)

print(
    selected_indices
)

print("✅ 15 internal images selected.")


# =============================================================================
# 8. VERIFY NO DATA LEAKAGE
# =============================================================================

print("\n" + "=" * 80)
print("🔐 DATA LEAKAGE CHECK")
print("=" * 80)

overlap = (
    set(selected_indices)
    .intersection(final_test_indices)
)

if len(overlap) > 0:

    raise ValueError(
        f"❌ Leakage detected: {overlap}"
    )

print(
    "Overlap with final 270 test set :",
    len(overlap)
)

print(
    "✅ No overlap with final evaluation set."
)


# =============================================================================
# 9. SHOW SELECTED CLASS DISTRIBUTION
# =============================================================================

print("\n" + "=" * 80)
print("📊 INTERNAL SUBSET DISTRIBUTION")
print("=" * 80)

selected_y = y[
    selected_indices
]

for class_index, class_name in enumerate(class_names):

    count = int(
        np.sum(
            selected_y == class_index
        )
    )

    print(
        f"{class_name:<15} : {count}"
    )


# =============================================================================
# 10. PREPARE INTERNAL IMAGES
# =============================================================================

print("\n" + "=" * 80)
print("🧪 APPLYING OFFICIAL TOMATO PREPROCESSING")
print("=" * 80)

X_internal = X[
    selected_indices
]

y_internal = y[
    selected_indices
]

# Official confirmed input contract:
# RGB
# 224 × 224
# float32
# 0–255

X_internal_model = X_internal.astype(
    np.float32
)

print(
    "Input shape :",
    X_internal_model.shape
)

print(
    "Input dtype :",
    X_internal_model.dtype
)

print(
    "Pixel range :",
    X_internal_model.min(),
    "to",
    X_internal_model.max()
)

if X_internal_model.shape != (
    15,
    224,
    224,
    3
):

    raise ValueError(
        f"Unexpected input shape: "
        f"{X_internal_model.shape}"
    )

if X_internal_model.dtype != np.float32:

    raise ValueError(
        "Input dtype is not float32."
    )

print(
    "✅ Official preprocessing applied."
)


# =============================================================================
# 11. LOAD FINAL MODEL
# =============================================================================

print("\n" + "=" * 80)
print("🧠 LOADING FINAL TOMATO CNN")
print("=" * 80)

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print(
    "Input shape :",
    model.input_shape
)

print(
    "Output shape:",
    model.output_shape
)

print(
    "Parameters  :",
    model.count_params()
)

if model.input_shape != (
    None,
    224,
    224,
    3
):

    raise ValueError(
        "Unexpected model input shape."
    )

if model.output_shape != (
    None,
    3
):

    raise ValueError(
        "Unexpected model output shape."
    )

print(
    "✅ Final Tomato CNN loaded."
)


# =============================================================================
# 12. RUN INTERNAL PREDICTIONS
# =============================================================================

print("\n" + "=" * 80)
print("🚀 RUNNING INTERNAL IMAGE PREDICTIONS")
print("=" * 80)

probabilities = model.predict(
    X_internal_model,
    batch_size=15,
    verbose=1
)

predicted_indices = np.argmax(
    probabilities,
    axis=1
)

predicted_classes = np.array([
    str(class_names[i])
    for i in predicted_indices
])

actual_classes = np.array([
    str(class_names[i])
    for i in y_internal
])

confidence = np.max(
    probabilities,
    axis=1
)

correct = (
    predicted_indices
    == y_internal
)


# =============================================================================
# 13. BUILD RESULT TABLE
# =============================================================================

results = pd.DataFrame({

    "dataset_index":
        selected_indices,

    "actual_class":
        actual_classes,

    "predicted_class":
        predicted_classes,

    "confidence":
        confidence,

    "correct":
        correct,

    "probability_healthy":
        probabilities[:, 0],

    "probability_early_blight":
        probabilities[:, 1],

    "probability_late_blight":
        probabilities[:, 2]

})


# =============================================================================
# 14. DISPLAY RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("📋 INTERNAL IMAGE-BY-IMAGE RESULTS")
print("=" * 80)

print(
    results[
        [
            "dataset_index",
            "actual_class",
            "predicted_class",
            "confidence",
            "correct"
        ]
    ].to_string(
        index=False
    )
)


# =============================================================================
# 15. OVERALL INTERNAL ACCURACY
# =============================================================================

correct_count = int(
    correct.sum()
)

total_count = len(
    correct
)

incorrect_count = (
    total_count
    - correct_count
)

accuracy = (
    correct_count
    / total_count
)

print("\n" + "=" * 80)
print("📈 INTERNAL RAW-DATA TEST RESULTS")
print("=" * 80)

print(
    f"Total images : {total_count}"
)

print(
    f"Correct      : {correct_count}"
)

print(
    f"Incorrect    : {incorrect_count}"
)

print(
    f"Accuracy     : "
    f"{accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)


# =============================================================================
# 16. CLASS-WISE PERFORMANCE
# =============================================================================

print("\n" + "=" * 80)
print("📊 INTERNAL CLASS-WISE PERFORMANCE")
print("=" * 80)

for class_name in class_names:

    class_mask = (
        actual_classes
        == str(class_name)
    )

    class_total = int(
        class_mask.sum()
    )

    class_correct = int(
        np.sum(
            correct[
                class_mask
            ]
        )
    )

    class_accuracy = (
        class_correct
        / class_total
    )

    print(
        f"{class_name:<15} : "
        f"{class_correct}/{class_total} "
        f"("
        f"{class_accuracy * 100:.2f}%"
        f")"
    )


# =============================================================================
# 17. CONFIDENCE ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("🎯 INTERNAL CONFIDENCE ANALYSIS")
print("=" * 80)

print(
    "Average confidence :",
    f"{confidence.mean():.4f}"
)

print(
    "Minimum confidence :",
    f"{confidence.min():.4f}"
)

print(
    "Maximum confidence :",
    f"{confidence.max():.4f}"
)


# =============================================================================
# 18. MISCLASSIFICATION CHECK
# =============================================================================

print("\n" + "=" * 80)
print("🔍 INTERNAL MISCLASSIFICATION CHECK")
print("=" * 80)

wrong = results[
    results["correct"] == False
]

print(
    "Misclassified images :",
    len(wrong)
)

if len(wrong) > 0:

    print("\nMisclassified cases:")

    print(
        wrong[
            [
                "dataset_index",
                "actual_class",
                "predicted_class",
                "confidence"
            ]
        ].to_string(
            index=False
        )
    )

else:

    print(
        "🎉 All selected internal images "
        "classified correctly."
    )


# =============================================================================
# 19. SAVE INTERNAL TEST RESULTS
# =============================================================================

RESULT_CSV = os.path.join(
    OUTPUT_DIR,
    "tomato_internal_raw_subset_test.csv"
)

SUMMARY_JSON = os.path.join(
    OUTPUT_DIR,
    "tomato_internal_raw_subset_summary.json"
)

results.to_csv(
    RESULT_CSV,
    index=False
)

summary = {

    "test_type":
        "Internal raw dataset subset",

    "selection":
        "5 images per class",

    "total_images":
        int(total_count),

    "correct":
        int(correct_count),

    "incorrect":
        int(incorrect_count),

    "accuracy":
        float(accuracy),

    "accuracy_percent":
        float(
            accuracy * 100
        ),

    "average_confidence":
        float(
            confidence.mean()
        ),

    "preprocessing":
        {
            "color": "RGB",
            "resize": "224x224",
            "dtype": "float32",
            "pixel_range": "0-255",
            "normalization": False
        },

    "excluded_final_evaluation_images":
        270,

    "model_modified":
        False,

    "dataset_modified":
        False,

    "training_performed":
        False
}

with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# =============================================================================
# 20. FINAL STATUS
# =============================================================================

print("\n" + "=" * 80)
print("🏁 INTERNAL RAW-DATA TEST COMPLETE")
print("=" * 80)

print(
    f"Images tested : {total_count}"
)

print(
    f"Accuracy      : "
    f"{accuracy * 100:.2f}%"
)

print(
    f"Correct       : {correct_count}"
)

print(
    f"Incorrect     : {incorrect_count}"
)

print("\n💾 Results saved:")
print(RESULT_CSV)

print("\n💾 Summary saved:")
print(SUMMARY_JSON)

print("\n✅ Model unchanged.")
print("✅ Dataset unchanged.")
print("✅ No training performed.")
print("✅ No files deleted.")
print("✅ Final 270 evaluation images excluded.")

print("=" * 80)

🍅 TOMATO CNN — INTERNAL RAW DATA SUBSET TEST
TensorFlow version : 2.20.0
Random seed        : 42

☁️ GOOGLE DRIVE CHECK
Mounted at /content/drive
✅ Google Drive accessible.

📁 FILE CHECK
✅ Tomato dataset found
✅ Tomato CNN found
✅ Final evaluation CSV found

📥 LOADING INTERNAL TOMATO DATASET
X shape         : (1800, 224, 224, 3)
X dtype         : uint8
y shape         : (1800,)
source shape    : (1800,)
class_names     : ['Healthy' 'Early_Blight' 'Late_Blight']
Raw pixel range : 0 to 255

Source distribution:
Internal_PlantVillage    900
External_Natural         900
Name: count, dtype: int64

Class distribution:
Healthy         : 600
Early_Blight    : 600
Late_Blight     : 600

✅ Internal dataset loaded.

🔒 EXCLUDING FINAL 270 EVALUATION IMAGES
Final evaluation images : 270
These images will NOT be sampled.
✅ Final evaluation set excluded.

🔎 IDENTIFYING AVAILABLE INTERNAL IMAGES
Total dataset images      : 1800
Final evaluation images   : 270
Remaining internal images : 1530

🎯 SELECT

In [ ]:
# =============================================================================
# 🍅 TOMATO CNN — FINAL INTERNAL vs EXTERNAL GENERALIZATION TEST
# =============================================================================
# PURPOSE:
#   1. Test 30 additional INTERNAL images
#      - Selected from existing tomato_processed_data.npz
#      - 10 Healthy + 10 Early_Blight + 10 Late_Blight
#      - Excludes the original 270 final evaluation images
#
#   2. Test 30 EXTERNAL / REAL-WORLD images
#      - Completely separate images from the project dataset
#      - Stored in 7th Test_Images/Tomato_RealWorld/
#      - 10 Healthy + 10 Early_Blight + 10 Late_Blight
#
#   3. Compare internal vs external performance
#
# OFFICIAL TOMATO PREPROCESSING:
#   RGB → 224×224 → float32 → 0–255
#
# ⚠️ READ-ONLY MODEL
# ⚠️ READ-ONLY DATASET
# ⚠️ NO TRAINING
# ⚠️ NO DATASET MODIFICATION
# ⚠️ NO FILE DELETION
# ⚠️ ORIGINAL 270 TEST IMAGES EXCLUDED FROM INTERNAL SUBSET
# =============================================================================

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from PIL import Image

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------------

N_PER_CLASS = 10

CLASS_NAMES = [
    "Healthy",
    "Early_Blight",
    "Late_Blight"
]

VALID_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
)

print("=" * 80)
print("🍅 TOMATO CNN — FINAL INTERNAL vs EXTERNAL TEST")
print("=" * 80)

print("TensorFlow version :", tf.__version__)
print("Random seed        :", SEED)
print("Images per class   :", N_PER_CLASS)
print("Total per test     :", N_PER_CLASS * len(CLASS_NAMES))


# =============================================================================
# 1. GOOGLE DRIVE
# =============================================================================

print("\n" + "=" * 80)
print("☁️ GOOGLE DRIVE CHECK")
print("=" * 80)

MYDRIVE = "/content/drive/MyDrive"

if not os.path.isdir(MYDRIVE):

    from google.colab import drive

    drive.mount("/content/drive")

print("✅ Google Drive accessible.")


# =============================================================================
# 2. PROJECT PATHS
# =============================================================================

PROJECT = (
    "/content/drive/MyDrive/"
    "Plant Disease Detection (Computer Vision)"
)

DATASET_PATH = os.path.join(
    PROJECT,
    "3rd Preprocessing",
    "tomato_processed_data.npz"
)

MODEL_PATH = os.path.join(
    PROJECT,
    "6th Trained_Model",
    "tomato_cnn_best.keras"
)

FINAL_EVAL_CSV = os.path.join(
    PROJECT,
    "5th Model_Evaluation",
    "tomato_test_predictions.csv"
)

EXTERNAL_DIR = os.path.join(
    PROJECT,
    "7th Test_Images",
    "Tomato_RealWorld"
)

OUTPUT_DIR = os.path.join(
    PROJECT,
    "5th Model_Evaluation",
    "Tomato_Image_Testing"
)

# Creating output directory does NOT modify dataset/model.
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("\nProject:")
print(PROJECT)

print("\nInternal dataset:")
print(DATASET_PATH)

print("\nFinal model:")
print(MODEL_PATH)

print("\nOriginal 270-image evaluation:")
print(FINAL_EVAL_CSV)

print("\nExternal / Real-World images:")
print(EXTERNAL_DIR)


# =============================================================================
# 3. FILE CHECK
# =============================================================================

print("\n" + "=" * 80)
print("📁 FILE CHECK")
print("=" * 80)

required_files = {
    "Tomato dataset": DATASET_PATH,
    "Tomato CNN": MODEL_PATH,
    "Final evaluation CSV": FINAL_EVAL_CSV
}

for name, path in required_files.items():

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"\n❌ {name} not found:\n{path}"
        )

    print(f"✅ {name} found")


if not os.path.isdir(EXTERNAL_DIR):

    raise FileNotFoundError(
        f"\n❌ External test directory not found:\n{EXTERNAL_DIR}"
    )

print("✅ External test directory found.")


# =============================================================================
# 4. LOAD INTERNAL DATASET
# =============================================================================

print("\n" + "=" * 80)
print("📥 LOADING INTERNAL TOMATO DATASET")
print("=" * 80)

data = np.load(
    DATASET_PATH,
    allow_pickle=True
)

X = data["X"]
y = data["y"]
class_names_dataset = data["class_names"]

print("X shape         :", X.shape)
print("X dtype         :", X.dtype)
print("y shape         :", y.shape)
print("class_names     :", class_names_dataset)

print(
    "Raw pixel range :",
    X.min(),
    "to",
    X.max()
)

if X.shape != (1800, 224, 224, 3):

    raise ValueError(
        f"Unexpected dataset shape: {X.shape}"
    )

if X.dtype != np.uint8:

    raise ValueError(
        f"Expected uint8 dataset, found {X.dtype}"
    )

print("✅ Internal dataset verified.")


# =============================================================================
# 5. DATASET CLASS DISTRIBUTION
# =============================================================================

print("\n" + "=" * 80)
print("📊 INTERNAL DATASET DISTRIBUTION")
print("=" * 80)

for i, class_name in enumerate(CLASS_NAMES):

    count = int(
        np.sum(y == i)
    )

    print(
        f"{class_name:<15} : {count}"
    )


# =============================================================================
# 6. RECOVER ORIGINAL 270 FINAL EVALUATION INDICES
# =============================================================================

print("\n" + "=" * 80)
print("🔒 RECOVERING ORIGINAL 270 EVALUATION INDICES")
print("=" * 80)

final_results = pd.read_csv(
    FINAL_EVAL_CSV
)

print(
    "Original evaluation rows :",
    len(final_results)
)

if "original_index" not in final_results.columns:

    raise ValueError(
        "\n❌ 'original_index' missing from final evaluation CSV."
    )

final_test_indices = (
    final_results["original_index"]
    .astype(int)
    .to_numpy()
)

if len(final_test_indices) != 270:

    raise ValueError(
        f"\n❌ Expected 270 final evaluation indices, "
        f"found {len(final_test_indices)}."
    )

if len(np.unique(final_test_indices)) != 270:

    raise ValueError(
        "\n❌ Duplicate indices found in final evaluation set."
    )

print(
    "Final evaluation images :",
    len(final_test_indices)
)

print(
    "Minimum index           :",
    final_test_indices.min()
)

print(
    "Maximum index           :",
    final_test_indices.max()
)

print("✅ Original 270-image evaluation set recovered.")


# =============================================================================
# 7. SELECT 30 ADDITIONAL INTERNAL IMAGES
# =============================================================================

print("\n" + "=" * 80)
print("🎯 SELECTING 30 ADDITIONAL INTERNAL IMAGES")
print("=" * 80)

final_test_set = set(
    final_test_indices.tolist()
)

internal_records = []

rng = np.random.default_rng(SEED)

for class_index, class_name in enumerate(CLASS_NAMES):

    all_class_indices = np.where(
        y == class_index
    )[0]

    available_indices = np.array([
        idx
        for idx in all_class_indices
        if idx not in final_test_set
    ])

    print(
        f"\n{class_name}"
    )

    print(
        "Available after exclusion :",
        len(available_indices)
    )

    if len(available_indices) < N_PER_CLASS:

        raise ValueError(
            f"Not enough available {class_name} images."
        )

    selected = rng.choice(
        available_indices,
        size=N_PER_CLASS,
        replace=False
    )

    selected = np.sort(selected)

    for idx in selected:

        internal_records.append({

            "dataset_index": int(idx),

            "actual_class": class_name,

            "source": "Internal_Dataset"
        })

internal_df = pd.DataFrame(
    internal_records
)

print("\nSelected internal images :", len(internal_df))

print("\nInternal distribution:")

print(
    internal_df["actual_class"]
    .value_counts()
    .reindex(
        CLASS_NAMES,
        fill_value=0
    )
)

if len(internal_df) != 30:

    raise ValueError(
        "❌ Internal test does not contain exactly 30 images."
    )

print("✅ 30 internal images selected.")


# =============================================================================
# 8. INTERNAL DATA LEAKAGE CHECK
# =============================================================================

print("\n" + "=" * 80)
print("🔐 INTERNAL DATA LEAKAGE CHECK")
print("=" * 80)

internal_indices = set(
    internal_df["dataset_index"]
    .astype(int)
    .tolist()
)

overlap = (
    internal_indices
    .intersection(final_test_set)
)

print(
    "Overlap with original 270 :",
    len(overlap)
)

if len(overlap) != 0:

    raise RuntimeError(
        "❌ DATA LEAKAGE DETECTED."
    )

print(
    "✅ Zero overlap with original 270 evaluation images."
)

print(
    "✅ Internal test is a separate subset."
)


# =============================================================================
# 9. PREPARE INTERNAL IMAGES
# =============================================================================

print("\n" + "=" * 80)
print("🧪 PREPARING INTERNAL IMAGES")
print("=" * 80)

X_internal_raw = X[
    internal_df["dataset_index"].to_numpy()
]

y_internal = np.array([
    CLASS_NAMES.index(class_name)
    for class_name
    in internal_df["actual_class"]
])

X_internal = X_internal_raw.astype(
    np.float32
)

print(
    "Shape       :",
    X_internal.shape
)

print(
    "Dtype       :",
    X_internal.dtype
)

print(
    "Pixel range :",
    X_internal.min(),
    "to",
    X_internal.max()
)

if X_internal.shape != (
    30,
    224,
    224,
    3
):

    raise ValueError(
        f"Unexpected internal shape: "
        f"{X_internal.shape}"
    )

print(
    "✅ Official preprocessing applied to internal images."
)


# =============================================================================
# 10. SCAN EXTERNAL DATA
# =============================================================================

print("\n" + "=" * 80)
print("🌍 SCANNING EXTERNAL / REAL-WORLD IMAGES")
print("=" * 80)

external_records = []

for class_name in CLASS_NAMES:

    class_dir = os.path.join(
        EXTERNAL_DIR,
        class_name
    )

    if not os.path.isdir(class_dir):

        raise FileNotFoundError(
            f"\n❌ Missing external class folder:\n{class_dir}"
        )

    files = sorted([
        filename
        for filename in os.listdir(class_dir)
        if filename.lower().endswith(
            VALID_EXTENSIONS
        )
    ])

    print(
        f"{class_name:<15} : {len(files)} images available"
    )

    if len(files) < N_PER_CLASS:

        raise ValueError(
            f"\n❌ Need at least {N_PER_CLASS} "
            f"external images for {class_name}, "
            f"but only {len(files)} found."
        )

    # Deterministic first 10 after alphabetical sorting.
    selected_files = files[:N_PER_CLASS]

    for filename in selected_files:

        external_records.append({

            "filepath": os.path.join(
                class_dir,
                filename
            ),

            "actual_class": class_name,

            "source": "External_RealWorld"
        })


external_df = pd.DataFrame(
    external_records
)

print(
    "\nExternal images selected :",
    len(external_df)
)

print("\nExternal distribution:")

print(
    external_df["actual_class"]
    .value_counts()
    .reindex(
        CLASS_NAMES,
        fill_value=0
    )
)

if len(external_df) != 30:

    raise ValueError(
        "❌ External test does not contain exactly 30 images."
    )

print(
    "✅ 30 external / real-world images selected."
)


# =============================================================================
# 11. LOAD EXTERNAL IMAGES
# =============================================================================

print("\n" + "=" * 80)
print("🖼️ PREPARING EXTERNAL / REAL-WORLD IMAGES")
print("=" * 80)

external_images = []
valid_external_records = []

for _, row in external_df.iterrows():

    filepath = row["filepath"]

    try:

        img = Image.open(filepath)

        # Convert to RGB
        img = img.convert("RGB")

        # Official model size
        img = img.resize(
            (224, 224),
            Image.Resampling.BILINEAR
        )

        # DO NOT normalize.
        img_array = np.asarray(
            img,
            dtype=np.float32
        )

        if img_array.shape != (
            224,
            224,
            3
        ):

            print(
                f"⚠️ Invalid shape: {filepath}"
            )

            continue

        external_images.append(
            img_array
        )

        valid_external_records.append(
            row
        )

    except Exception as e:

        print(
            f"⚠️ Could not read: {filepath}"
        )

        print(
            "   Reason:",
            e
        )


if len(external_images) != 30:

    raise ValueError(
        f"\n❌ Expected 30 readable external images, "
        f"but got {len(external_images)}."
    )

X_external = np.stack(
    external_images
)

external_valid_df = pd.DataFrame(
    valid_external_records
)

y_external = np.array([
    CLASS_NAMES.index(class_name)
    for class_name
    in external_valid_df["actual_class"]
])

print(
    "\nExternal shape       :",
    X_external.shape
)

print(
    "External dtype       :",
    X_external.dtype
)

print(
    "External pixel range:",
    X_external.min(),
    "to",
    X_external.max()
)

print(
    "✅ Official preprocessing applied to external images."
)


# =============================================================================
# 12. LOAD FINAL TOMATO CNN
# =============================================================================

print("\n" + "=" * 80)
print("🧠 LOADING FINAL TOMATO CNN")
print("=" * 80)

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print(
    "Input shape :",
    model.input_shape
)

print(
    "Output shape:",
    model.output_shape
)

print(
    "Parameters  :",
    model.count_params()
)

if model.input_shape != (
    None,
    224,
    224,
    3
):

    raise ValueError(
        f"Unexpected model input: "
        f"{model.input_shape}"
    )

if model.output_shape != (
    None,
    3
):

    raise ValueError(
        f"Unexpected model output: "
        f"{model.output_shape}"
    )

print(
    "✅ Final Tomato CNN loaded."
)


# =============================================================================
# 13. INTERNAL PREDICTIONS
# =============================================================================

print("\n" + "=" * 80)
print("🚀 RUNNING INTERNAL PREDICTIONS")
print("=" * 80)

internal_prob = model.predict(
    X_internal,
    batch_size=16,
    verbose=1
)

internal_pred_indices = np.argmax(
    internal_prob,
    axis=1
)

internal_predicted = [
    CLASS_NAMES[i]
    for i in internal_pred_indices
]

internal_confidence = np.max(
    internal_prob,
    axis=1
)

internal_results = internal_df.copy()

internal_results[
    "predicted_class"
] = internal_predicted

internal_results[
    "confidence"
] = internal_confidence

internal_results[
    "correct"
] = (
    internal_results["actual_class"]
    == internal_results["predicted_class"]
)


# =============================================================================
# 14. EXTERNAL PREDICTIONS
# =============================================================================

print("\n" + "=" * 80)
print("🚀 RUNNING EXTERNAL / REAL-WORLD PREDICTIONS")
print("=" * 80)

external_prob = model.predict(
    X_external,
    batch_size=16,
    verbose=1
)

external_pred_indices = np.argmax(
    external_prob,
    axis=1
)

external_predicted = [
    CLASS_NAMES[i]
    for i in external_pred_indices
]

external_confidence = np.max(
    external_prob,
    axis=1
)

external_results = external_valid_df.copy()

external_results[
    "predicted_class"
] = external_predicted

external_results[
    "confidence"
] = external_confidence

external_results[
    "correct"
] = (
    external_results["actual_class"]
    == external_results["predicted_class"]
)


# =============================================================================
# 15. INTERNAL RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("📋 INTERNAL IMAGE RESULTS")
print("=" * 80)

print(
    internal_results[
        [
            "dataset_index",
            "actual_class",
            "predicted_class",
            "confidence",
            "correct"
        ]
    ].to_string(index=False)
)


# =============================================================================
# 16. EXTERNAL RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("📋 EXTERNAL / REAL-WORLD IMAGE RESULTS")
print("=" * 80)

print(
    external_results[
        [
            "filepath",
            "actual_class",
            "predicted_class",
            "confidence",
            "correct"
        ]
    ].to_string(index=False)
)


# =============================================================================
# 17. CALCULATE INTERNAL METRICS
# =============================================================================

internal_correct = int(
    internal_results["correct"].sum()
)

internal_total = len(
    internal_results
)

internal_incorrect = (
    internal_total
    - internal_correct
)

internal_accuracy = (
    internal_correct
    / internal_total
)


# =============================================================================
# 18. CALCULATE EXTERNAL METRICS
# =============================================================================

external_correct = int(
    external_results["correct"].sum()
)

external_total = len(
    external_results
)

external_incorrect = (
    external_total
    - external_correct
)

external_accuracy = (
    external_correct
    / external_total
)


# =============================================================================
# 19. INTERNAL SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("📈 INTERNAL TEST RESULTS")
print("=" * 80)

print(
    f"Total images : {internal_total}"
)

print(
    f"Correct      : {internal_correct}"
)

print(
    f"Incorrect    : {internal_incorrect}"
)

print(
    f"Accuracy     : "
    f"{internal_accuracy:.4f} "
    f"({internal_accuracy * 100:.2f}%)"
)


# =============================================================================
# 20. EXTERNAL SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("📈 EXTERNAL / REAL-WORLD TEST RESULTS")
print("=" * 80)

print(
    f"Total images : {external_total}"
)

print(
    f"Correct      : {external_correct}"
)

print(
    f"Incorrect    : {external_incorrect}"
)

print(
    f"Accuracy     : "
    f"{external_accuracy:.4f} "
    f"({external_accuracy * 100:.2f}%)"
)


# =============================================================================
# 21. CLASS-WISE INTERNAL PERFORMANCE
# =============================================================================

print("\n" + "=" * 80)
print("📊 INTERNAL CLASS-WISE PERFORMANCE")
print("=" * 80)

internal_class_metrics = []

for class_name in CLASS_NAMES:

    subset = internal_results[
        internal_results["actual_class"]
        == class_name
    ]

    correct = int(
        subset["correct"].sum()
    )

    total = len(subset)

    acc = correct / total

    internal_class_metrics.append({

        "test_source": "Internal",

        "class": class_name,

        "correct": correct,

        "total": total,

        "accuracy_percent": acc * 100
    })

    print(
        f"{class_name:<15} : "
        f"{correct}/{total} "
        f"({acc * 100:.2f}%)"
    )


# =============================================================================
# 22. CLASS-WISE EXTERNAL PERFORMANCE
# =============================================================================

print("\n" + "=" * 80)
print("📊 EXTERNAL CLASS-WISE PERFORMANCE")
print("=" * 80)

external_class_metrics = []

for class_name in CLASS_NAMES:

    subset = external_results[
        external_results["actual_class"]
        == class_name
    ]

    correct = int(
        subset["correct"].sum()
    )

    total = len(subset)

    acc = correct / total

    external_class_metrics.append({

        "test_source": "External",

        "class": class_name,

        "correct": correct,

        "total": total,

        "accuracy_percent": acc * 100
    })

    print(
        f"{class_name:<15} : "
        f"{correct}/{total} "
        f"({acc * 100:.2f}%)"
    )


# =============================================================================
# 23. CONFIDENCE COMPARISON
# =============================================================================

print("\n" + "=" * 80)
print("🎯 CONFIDENCE COMPARISON")
print("=" * 80)

internal_avg_conf = float(
    internal_results["confidence"].mean()
)

external_avg_conf = float(
    external_results["confidence"].mean()
)

internal_min_conf = float(
    internal_results["confidence"].min()
)

external_min_conf = float(
    external_results["confidence"].min()
)

internal_max_conf = float(
    internal_results["confidence"].max()
)

external_max_conf = float(
    external_results["confidence"].max()
)

print(
    f"Internal average confidence : "
    f"{internal_avg_conf:.4f}"
)

print(
    f"External average confidence : "
    f"{external_avg_conf:.4f}"
)

print(
    f"Internal minimum confidence : "
    f"{internal_min_conf:.4f}"
)

print(
    f"External minimum confidence : "
    f"{external_min_conf:.4f}"
)

print(
    f"Internal maximum confidence : "
    f"{internal_max_conf:.4f}"
)

print(
    f"External maximum confidence : "
    f"{external_max_conf:.4f}"
)


# =============================================================================
# 24. FINAL COMPARISON
# =============================================================================

print("\n" + "=" * 80)
print("⚖️ INTERNAL vs EXTERNAL COMPARISON")
print("=" * 80)

accuracy_gap = (
    internal_accuracy
    - external_accuracy
)

print(
    f"Internal accuracy : "
    f"{internal_accuracy * 100:.2f}%"
)

print(
    f"External accuracy : "
    f"{external_accuracy * 100:.2f}%"
)

print(
    f"Accuracy gap      : "
    f"{accuracy_gap * 100:.2f} percentage points"
)

if accuracy_gap > 0:

    print(
        "\n📌 Internal performance is higher "
        "than external performance."
    )

elif accuracy_gap < 0:

    print(
        "\n📌 External performance is higher "
        "than internal performance."
    )

else:

    print(
        "\n📌 Internal and external performance "
        "are identical."
    )


# =============================================================================
# 25. MISCLASSIFICATION SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🔍 MISCLASSIFICATION SUMMARY")
print("=" * 80)

internal_wrong = internal_results[
    internal_results["correct"] == False
]

external_wrong = external_results[
    external_results["correct"] == False
]

print(
    "Internal misclassifications :",
    len(internal_wrong)
)

print(
    "External misclassifications :",
    len(external_wrong)
)

if len(internal_wrong) > 0:

    print("\nInternal mistakes:")

    print(
        internal_wrong[
            [
                "dataset_index",
                "actual_class",
                "predicted_class",
                "confidence"
            ]
        ].to_string(index=False)
    )

else:

    print(
        "🎉 No internal misclassifications."
    )


if len(external_wrong) > 0:

    print("\nExternal mistakes:")

    print(
        external_wrong[
            [
                "actual_class",
                "predicted_class",
                "confidence"
            ]
        ].to_string(index=False)
    )

else:

    print(
        "🎉 No external misclassifications."
    )


# =============================================================================
# 26. COMBINED CLASS METRICS
# =============================================================================

combined_class_metrics = pd.DataFrame(
    internal_class_metrics
    + external_class_metrics
)

print("\n" + "=" * 80)
print("📊 FINAL CLASS-WISE COMPARISON")
print("=" * 80)

print(
    combined_class_metrics.to_string(
        index=False
    )
)


# =============================================================================
# 27. SAVE INTERNAL RESULTS
# =============================================================================

INTERNAL_CSV = os.path.join(
    OUTPUT_DIR,
    "tomato_internal_final_30_test.csv"
)

internal_results.to_csv(
    INTERNAL_CSV,
    index=False
)


# =============================================================================
# 28. SAVE EXTERNAL RESULTS
# =============================================================================

EXTERNAL_CSV = os.path.join(
    OUTPUT_DIR,
    "tomato_external_final_30_test.csv"
)

external_results.to_csv(
    EXTERNAL_CSV,
    index=False
)


# =============================================================================
# 29. SAVE COMBINED SUMMARY
# =============================================================================

SUMMARY_JSON = os.path.join(
    OUTPUT_DIR,
    "tomato_internal_vs_external_final_summary.json"
)

summary = {

    "model": MODEL_PATH,

    "official_preprocessing": {
        "color": "RGB",
        "resize": "224x224",
        "dtype": "float32",
        "pixel_range": "0-255",
        "normalization": False
    },

    "original_evaluation": {
        "images": 270,
        "accuracy_percent": 97.78,
        "correct": 264,
        "incorrect": 6
    },

    "internal_test": {

        "description":
            "Small subset selected from existing "
            "tomato_processed_data.npz",

        "images_per_class":
            N_PER_CLASS,

        "total_images":
            internal_total,

        "correct":
            internal_correct,

        "incorrect":
            internal_incorrect,

        "accuracy":
            float(internal_accuracy),

        "accuracy_percent":
            float(internal_accuracy * 100),

        "average_confidence":
            internal_avg_conf,

        "minimum_confidence":
            internal_min_conf,

        "maximum_confidence":
            internal_max_conf,

        "excluded_original_270":
            True,

        "overlap_with_original_270":
            0
    },

    "external_test": {

        "description":
            "Independent real-world / Internet images "
            "stored outside the project dataset",

        "images_per_class":
            N_PER_CLASS,

        "total_images":
            external_total,

        "correct":
            external_correct,

        "incorrect":
            external_incorrect,

        "accuracy":
            float(external_accuracy),

        "accuracy_percent":
            float(external_accuracy * 100),

        "average_confidence":
            external_avg_conf,

        "minimum_confidence":
            external_min_conf,

        "maximum_confidence":
            external_max_conf
    },

    "comparison": {

        "internal_accuracy_percent":
            float(internal_accuracy * 100),

        "external_accuracy_percent":
            float(external_accuracy * 100),

        "accuracy_gap_percentage_points":
            float(accuracy_gap * 100)
    },

    "model_modified": False,

    "dataset_modified": False,

    "training_performed": False
}

with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# =============================================================================
# 30. FINAL STATUS
# =============================================================================

print("\n" + "=" * 80)
print("🏁 FINAL INTERNAL vs EXTERNAL TEST COMPLETE")
print("=" * 80)

print(
    f"Original evaluation : 270 images → 97.78%"
)

print(
    f"Internal test       : "
    f"{internal_total} images → "
    f"{internal_accuracy * 100:.2f}%"
)

print(
    f"External test       : "
    f"{external_total} images → "
    f"{external_accuracy * 100:.2f}%"
)

print(
    f"Accuracy gap        : "
    f"{accuracy_gap * 100:.2f} percentage points"
)

print("\n💾 Results saved:")

print(
    "\nInternal CSV:"
)

print(
    INTERNAL_CSV
)

print(
    "\nExternal CSV:"
)

print(
    EXTERNAL_CSV
)

print(
    "\nCombined summary:"
)

print(
    SUMMARY_JSON
)

print("\n" + "-" * 80)

print("✅ Model unchanged.")
print("✅ Training dataset unchanged.")
print("✅ No training performed.")
print("✅ No files deleted.")
print("✅ Original 270 evaluation images excluded from internal subset.")
print("✅ Internal test = existing dataset subset.")
print("✅ External test = independent real-world images.")

print("=" * 80)

🍅 TOMATO CNN — FINAL INTERNAL vs EXTERNAL TEST
TensorFlow version : 2.20.0
Random seed        : 42
Images per class   : 10
Total per test     : 30

☁️ GOOGLE DRIVE CHECK
✅ Google Drive accessible.

Project:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)

Internal dataset:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/tomato_processed_data.npz

Final model:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/tomato_cnn_best.keras

Original 270-image evaluation:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/5th Model_Evaluation/tomato_test_predictions.csv

External / Real-World images:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/7th Test_Images/Tomato_RealWorld

📁 FILE CHECK
✅ Tomato dataset found
✅ Tomato CNN found
✅ Final evaluation CSV found
✅ External test directory found.

📥 LOADING INTERNAL TOMATO DATASET
X shape         : (1800, 224, 224, 3)
X dtype        

2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step

🚀 RUNNING EXTERNAL / REAL-WORLD PREDICTIONS
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 516ms/step

📋 INTERNAL IMAGE RESULTS
 dataset_index actual_class predicted_class  confidence  correct
            50      Healthy         Healthy    0.999882     True
            51      Healthy         Healthy    0.996578     True
            56      Healthy         Healthy    0.991692     True
           125      Healthy         Healthy    0.999655     True
           262      Healthy         Healthy    0.999885     True
           265      Healthy         Healthy    0.999889     True
           988      Healthy         Healthy    0.999817     True
          1013      Healthy         Healthy    0.998777     True
          1055      Healthy         Healthy    0.987468     True
          1104      Healthy         Healthy    0.999390     True
           406 Early_Blight    Early_Blight    0.689205     True
           521 Early_Blight    Early_Blight    0.889105     True
         